## 1 - Extract - Leitura dos dados

In [3]:
from dotenv import load_dotenv
import os
import pandas as pd
import chardet
from pathlib import Path

#load_dotenv() não lembro porque coloquei

RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")

if RAW_DATA_PATH is None:
    raise ValueError("Caminho do arquivo .env vazio.")

csv_path = Path(RAW_DATA_PATH)


#Identificador do Encoding
with open(csv_path, "rb") as f:

    # Lê TODO o conteúdo do arquivo como uma sequência de bytes
    # e utiliza a biblioteca chardet para detectar automaticamente o encoding
    resultado = chardet.detect(f.read())
print(resultado)




{'encoding': 'ascii', 'confidence': 1.0, 'language': 'en', 'mime_type': 'text/plain'}


In [4]:
#Leitura arquivo
df = pd.read_csv(csv_path,encoding="latin-1", sep=",") 

### 2 - Insperção dos dados

    - Etapa de verificação/diagnóstico dos dados para tratamento e analise.

In [5]:
print(f'Quantidade de linhas: {df.shape[0]}\nQuantidade de Colunas: {df.shape[1]}')
print('-'*50)
print(df.dtypes)
print('-'*50)
print(df.isnull().sum())
print('-'*50)
print(f'Valores duplicados: {df.duplicated().sum()}')
print('-'*50)
df.describe(include='all')
 

Quantidade de linhas: 541909
Quantidade de Colunas: 8
--------------------------------------------------
InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object
--------------------------------------------------
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
--------------------------------------------------
Valores duplicados: 5268
--------------------------------------------------


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909,541909,540455,541909.000000,541909,541909.000000,406829.000000,541909
unique,25900,4070,4223,NaN,23260,NaN,NaN,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,10/31/2011 14:41,NaN,NaN,United Kingdom
freq,1114,2313,2369,NaN,1114,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,NaN,4.611114,15287.690570,NaN
std,NaN,NaN,NaN,218.081158,NaN,96.759853,1713.600303,NaN
min,NaN,NaN,NaN,-80995.000000,NaN,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,NaN,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,NaN,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,NaN,4.130000,16791.000000,NaN


#### Valores Duplicados

In [6]:
print(f'Valores duplicados: {df.duplicated().sum()}')

filtro_duplicados = df.duplicated(subset=['InvoiceNo', 'StockCode'], keep=False)
filtro_duplicados.head(20)

df_duplicados = df[filtro_duplicados]
df_duplicados.head(20).reset_index(drop=True)


'''
- Decisão sobre linhas duplicadas:

Foram identificadas 5.268 linhas com combinação duplicada de InvoiceNo + StockCode.
Essas duplicatas podem ser originadas de falhas no sistema de registro (duplo clique no lançamento),
reenvio de requisição ou erro de importação, situações em que o mesmo item aparece mais de uma vez
na mesma nota. Optou-se por manter a primeira ocorrência (keep='first'), assumindo que o primeiro
registro representa o lançamento original. Essa decisão pode introduzir pequenas distorções em
análises de quantidade e receita, mas preserva o volume de transações no dataset. 
Por a quantidade ser abaixo de 1% da quantidade total de linhas, não deve impactar o resultado.
'''

df = df.drop_duplicates(subset=['InvoiceNo', 'StockCode'], keep='first')
df.shape

Valores duplicados: 5268


(531225, 8)

In [7]:
# InvoiceDate: str → datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# CustomerID: float → string
# Primeiro preenche os nulos, depois converte
# (converter float com NaN direto para int gera erro)
df['CustomerID'] = df['CustomerID'].fillna('Anonimo')
df['CustomerID'] = df['CustomerID'].apply(
    lambda x: str(int(x)) if x != 'Anonimo' else x
)

In [8]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France


##

# Old

## Análise Exploratória de Dados (EDA) 


In [3]:
print(df.sample(5))


       InvoiceNo StockCode                     Description  Quantity  \
338767    566564     22616      PACK OF 12 LONDON TISSUES         24   
348752    567462     23318  BOX OF 6 MINI VINTAGE CRACKERS         2   
472678    576695     16237            SLEEPING CAT ERASERS         1   
463070    576053     48111           DOORMAT 3 SMILEY CATS         5   
1559      536544     22359          GLASS JAR KINGS CHOICE         2   

             InvoiceDate  UnitPrice  CustomerID         Country  
338767   9/13/2011 12:26       0.39     15522.0  United Kingdom  
348752   9/20/2011 12:35       2.49     14446.0  United Kingdom  
472678  11/16/2011 12:25       0.21     16469.0  United Kingdom  
463070  11/13/2011 14:53       8.25     16726.0  United Kingdom  
1559     12/1/2010 14:32       5.91         NaN  United Kingdom  


In [5]:
df.shape #Quantidade de linhas e colunas

(541909, 8)

In [6]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [7]:
df.isnull().sum().sum()

np.int64(136534)

In [8]:

df.duplicated().sum()

np.int64(5268)

In [9]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [11]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

## Tratativas

### Tratando dados str e colunas

In [1]:
df.columns

NameError: name 'df' is not defined

In [ ]:
### Renomeando Columns

colunas_traduzidas = {
    "InvoiceNo": "numero_fatura",
    "StockCode": "codigo_produto",
    "Description": "descricao",
    "Quantity": "quantidade",
    "InvoiceDate": "data_fatura",
    "UnitPrice": "preco_unitario",
    "CustomerID": "id_cliente",
    "Country": "pais"
}


df = df.rename(columns=colunas_traduzidas)
df.columns

Index(['numero_fatura', 'codigo_produto', 'descricao', 'quantidade',
       'data_fatura', 'preco_unitario', 'id_cliente', 'pais'],
      dtype='str')

In [ ]:
### Tratamento coluna País/Country

#Unifica os valores para visualização
print(df['pais'].unique()) 

# Tratamento: Retira os espaços vázios, deixa a formatação Title e substitui os valores srt
df['pais'] = (df['pais'].str.strip().str.title().replace({
        "Eire": "Ireland",
        "Usa": "United States",
        "Rsa": "South Africa",
        "Unspecified": None,
        "European Community": "Europe"
    })) 


print(df['pais'].isnull().sum())

df = df.dropna(subset=['pais'])

df['pais'] = df['pais'].astype('category')

<StringArray>
[      'United Kingdom',               'France',            'Australia',
          'Netherlands',              'Germany',               'Norway',
                 'EIRE',          'Switzerland',                'Spain',
               'Poland',             'Portugal',                'Italy',
              'Belgium',            'Lithuania',                'Japan',
              'Iceland',      'Channel Islands',              'Denmark',
               'Cyprus',               'Sweden',              'Austria',
               'Israel',              'Finland',              'Bahrain',
               'Greece',            'Hong Kong',            'Singapore',
              'Lebanon', 'United Arab Emirates',         'Saudi Arabia',
       'Czech Republic',               'Canada',          'Unspecified',
               'Brazil',                  'USA',   'European Community',
                'Malta',                  'RSA']
Length: 38, dtype: str
446


In [ ]:
### Tratamento coluna data_fatura

# Converte a coluna 'data_fatura' para o tipo datetime
# errors="coerce" garante que valores inválidos sejam transformados em NaT (Not a Time),
# evitando que o código quebre durante a execução
df['data_fatura'] = pd.to_datetime(df['data_fatura'], errors="coerce")




In [ ]:
df_filtro = df[
    df["pais"].isna() |
    df["descricao"].isna() |
    df["id_cliente"].isna()
]

df_filtro.info()

<class 'pandas.DataFrame'>
Index: 134878 entries, 622 to 541540
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   numero_fatura   134878 non-null  str           
 1   codigo_produto  134878 non-null  str           
 2   descricao       133424 non-null  str           
 3   quantidade      134878 non-null  int64         
 4   data_fatura     134878 non-null  datetime64[us]
 5   preco_unitario  134878 non-null  float64       
 6   id_cliente      0 non-null       float64       
 7   pais            134878 non-null  category      
dtypes: category(1), datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 8.4 MB


In [ ]:
df_filtro.reset_index(drop=True)

df_filtro_preco = df[df['preco_unitario'] == 0]

df_filtro_preco

,numero_fatura,codigo_produto,descricao,quantidade,data_fatura,preco_unitario,id_cliente,pais
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom
538504,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom
538505,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom
538554,581408,85175,NaN,20,2011-12-08 14:06:00,0.0,NaN,United Kingdom


In [ ]:
df = df.dropna(
    subset=["pais", "descricao", "id_cliente"],
    how="any"  # padrão, mas bom deixar explícito
)

## Outros

In [19]:

'''
df.isnull().sum() #FAZER TRATAMENTO DOS NULLS

df = df.drop_duplicates(subset=["CustomerID"])

print(f"Quantidade de linhas: {df.shape[0]}\nQuantidade de Colunas: {df.shape[1]} ") 

df = df.drop_duplicates().reset_index(drop=True)
df 
'''

'\ndf.isnull().sum() #FAZER TRATAMENTO DOS NULLS\n\ndf = df.drop_duplicates(subset=["CustomerID"])\n\nprint(f"Quantidade de linhas: {df.shape[0]}\nQuantidade de Colunas: {df.shape[1]} ") \n\ndf = df.drop_duplicates().reset_index(drop=True)\ndf \n'

In [20]:
df.shape

#df.info()

(406585, 8)